# 📈 MODEL 1: Dự báo Suy giảm Hiệu năng Chuỗi Thời gian (5-Minute Time-Series Forecaster)

---
### 📌 Nguyên tắc Thiết kế (Separation of Concerns):
- **Model 1 chỉ làm đúng 1 nhiệm vụ duy nhất:** Quan sát chuỗi thời gian cửa sổ trượt $[t-W \dots t]$ (gồm giá trị hiện tại, trung bình trượt `roll_mean`, độ lệch chuẩn `roll_std`, gia tốc tức thời `delta1`, độ dốc xu hướng `trend_slope` và tỷ lệ duy trì `sustained_ratio`) để dự đoán xác suất suy giảm hiệu năng trong **5 phút tới** (`degraded_forecast_5m`: `0` hoặc `1`).
- **Không nhúng cứng quy tắc gọi Model 2 vào trong Model 1:** Model 1 chỉ xuất ra xác suất dự báo suy giảm $P(\text{degraded\_5m}) \in [0, 1]$ và điểm bất thường. Còn việc kiểm tra điều kiện khi nào kích hoạt Model 2 (ngưỡng bao nhiêu %, kéo dài bao lâu) do **Policy Engine của Hệ thống** đảm nhiệm để quản trị viên có thể tinh chỉnh linh hoạt trong tương lai mà không cần train lại model.

### 📌 Dữ liệu & Đầu ra:
- **File Dataset:** `dataset_model1_timeseries.csv` (`700 mẫu × 84 đặc trưng Sliding Window` + 1 cột nhãn mục tiêu `degraded_forecast_5m`).
- **File Model xuất ra:** `anomaly_detector.pkl`

In [ ]:
import glob
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    IsolationForest,
    RandomForestClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 11

## 1. Nạp Dataset Chuỗi thời gian (`dataset_model1_timeseries.csv`)

In [ ]:
candidates = (
    glob.glob('/kaggle/input/**/dataset_model1_timeseries.csv', recursive=True)
    + ['../../dataset/dataset_model1_timeseries.csv', '../dataset/dataset_model1_timeseries.csv', 'dataset_model1_timeseries.csv']
)
csv_path = next((p for p in candidates if os.path.exists(p)), None)
assert csv_path is not None, 'Không tìm thấy dataset_model1_timeseries.csv! Hãy upload file này lên Kaggle.'

df_m1 = pd.read_csv(csv_path).replace([np.inf, -np.inf], np.nan).fillna(0.0)

TARGET_COL = 'degraded_forecast_5m'
META_COLS = {'timestamp', 'datetime', TARGET_COL}
feature_cols = [c for c in df_m1.columns if c not in META_COLS]

KEY_TIMESERIES_KPIS = [
    'wan1_rx_bytes_rate', 'wan1_tx_bytes_rate', 'branch_tcp_retrans_rate',
    'wan_tcp_syn_retrans_rate', 'wan_tcp_timeouts_rate', 'fw_rx_pkts_rate',
    'fw_avg_pkt_size', 'client_lan_tx_pkts_rate', 'srv_tcp_out_rsts_rate',
    'srv_tcp_passive_opens_rate', 'srv_cpu_usage', 'nginx_req_rate',
    'api_req_rate', 'api_latency_avg'
]

print(f'✅ Đã nạp Dataset Model 1: {csv_path}')
print(f'📊 Kích thước: {df_m1.shape[0]} mẫu x {len(feature_cols)} đặc trưng cửa sổ trượt (Sliding Window)')
print(f'🎯 Nhãn mục tiêu duy nhất: {TARGET_COL}')
display(df_m1.head())

In [ ]:
# Phân bố nhãn mục tiêu degraded_forecast_5m
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

counts = df_m1[TARGET_COL].map({0: '0: Ổn định / Tự khỏi', 1: '1: Suy giảm 5 phút tới'}).value_counts()
sns.barplot(x=counts.values, y=counts.index, palette=['#e74c3c', '#2ecc71'], ax=ax[0])
ax[0].set_title('Số lượng mẫu theo Nhãn Dự báo Suy giảm 5 Phút', fontweight='bold')
for i, v in enumerate(counts.values):
    ax[0].text(v + 3, i, str(v), va='center', fontweight='bold')

ax[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=['#e74c3c', '#2ecc71'])
ax[1].set_title('Tỷ lệ Phân bố Nhãn (Binary Time-Series Target)', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Trực quan hóa Đặc trưng Chuỗi thời gian (Sliding Window Trends)

In [ ]:
sample_slice = df_m1.iloc[:200].reset_index(drop=True)
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax[0].plot(sample_slice.index, sample_slice['srv_cpu_usage_current'], label='CPU Server hiện tại (%)', color='#2c3e50')
ax[0].plot(sample_slice.index, sample_slice['srv_cpu_usage_roll_mean'], label='Trung bình trượt (Rolling Mean)', color='#e67e22', linestyle='--')
ax[0].set_ylabel('CPU (%)')
ax[0].set_title('Chuỗi thời gian Chỉ số Dịch vụ & Nhãn Dự báo Suy giảm 5 Phút (degraded_forecast_5m)', fontweight='bold')
ax[0].legend(loc='upper right')

ax[1].fill_between(sample_slice.index, sample_slice[TARGET_COL], color='#e74c3c', alpha=0.45, step='mid', label='Nhãn mục tiêu: degraded_forecast_5m (0/1)')
ax[1].set_ylabel('Target (0/1)')
ax[1].set_xlabel('Bước thời gian (Time Steps)')
ax[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 3. Huấn luyện & Đối sánh các Thuật toán cho Model 1
Đối sánh 4 thuật toán dự báo trên tập đặc trưng chuỗi thời gian:
- **Random Forest (`RandomForest_TS`)**
- **Extra Trees (`ExtraTrees_TS`)**
- **Gradient Boosting (`GradientBoosting_TS`)**
- **XGBoost (`XGBoost_TS`)**

In [ ]:
X = df_m1[feature_cols].values
y = df_m1[TARGET_COL].values.astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

scaler = RobustScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
X_all_s = RobustScaler().fit_transform(X)

candidates = {
    'RandomForest_TS': RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1),
    'ExtraTrees_TS': ExtraTreesClassifier(n_estimators=300, max_depth=14, class_weight='balanced', random_state=42, n_jobs=-1),
    'GradientBoosting_TS': GradientBoostingClassifier(n_estimators=200, learning_rate=0.08, max_depth=4, random_state=42),
}

try:
    from xgboost import XGBClassifier
    candidates['XGBoost_TS'] = XGBClassifier(
        n_estimators=250, learning_rate=0.08, max_depth=5, subsample=0.9, colsample_bytree=0.9,
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
except ImportError:
    pass

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
trained_models = {}
plt.figure(figsize=(8, 6))

for name, clf in candidates.items():
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_te_s)
    probs = clf.predict_proba(X_te_s)[:, 1]
    cv_scores = cross_val_score(clf, X_all_s, y, cv=cv, scoring='f1', n_jobs=-1)
    trained_models[name] = (clf, preds, probs)
    
    auc = roc_auc_score(y_te, probs)
    fpr, tpr, _ = roc_curve(y_te, probs)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.4f})')
    
    results.append({
        'Algorithm': name,
        'Accuracy': accuracy_score(y_te, preds),
        'Precision': precision_score(y_te, preds),
        'Recall': recall_score(y_te, preds),
        'F1-Score': f1_score(y_te, preds),
        'ROC-AUC': auc,
        '5-Fold CV F1': np.mean(cv_scores),
    })

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title('ROC Curve - Dự báo Suy giảm Chuỗi thời gian 5 Phút Tới (Model 1)', fontweight='bold')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend(loc='lower right')
plt.show()

res_df = pd.DataFrame(results).sort_values(['F1-Score', 'ROC-AUC'], ascending=False)
display(res_df.round(4))

## 4. Báo cáo Đánh giá Chi tiết, Confusion Matrix & Top Feature Importances

In [ ]:
best_name = res_df.iloc[0]['Algorithm']
best_clf, best_preds, best_probs = trained_models[best_name]

print(f'🏆 Mô hình Dự báo Chuỗi thời gian (Model 1) tốt nhất: {best_name}')
print(classification_report(y_te, best_preds, target_names=['Ổn định / Tự khỏi (0)', 'Suy giảm 5 phút tới (1)'], digits=4))

cm = confusion_matrix(y_te, best_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=['Dự báo Ổn định (0)', 'Dự báo Suy giảm (1)'], yticklabels=['Thực tế Ổn định (0)', 'Thực tế Suy giảm (1)'])
plt.title(f'Confusion Matrix - Model 1 ({best_name})', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
importances = getattr(best_clf, 'feature_importances_', np.zeros(len(feature_cols)))
imp_df = pd.DataFrame({'Time-Series Feature': feature_cols, 'Importance': importances}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(11, 6))
sns.barplot(x='Importance', y='Time-Series Feature', data=imp_df, palette='rocket')
plt.title(f'Top 15 Đặc trưng Chuỗi thời gian Quan trọng nhất ({best_name})', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Đóng gói & Xuất File `anomaly_detector.pkl` (Model 1)

In [ ]:
final_scaler = RobustScaler()
X_all_scaled = final_scaler.fit_transform(X)

# Kèm thêm IsolationForest học trên mẫu ổn định để tính điểm bất thường tức thời (unsupervised anomaly score)
final_iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=42, n_jobs=-1)
final_iso.fit(X_all_scaled[y == 0])

# Mô hình dự báo chuỗi thời gian 5 phút tới
final_forecaster = RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
final_forecaster.fit(X_all_scaled, y)

artifact_m1 = {
    'model_type': 'RandomForest_TS_Forecaster',
    'scaler': final_scaler,
    'model': final_forecaster,
    'unsupervised_detector': final_iso,
    'feature_names': feature_cols,
    'key_kpis': KEY_TIMESERIES_KPIS,
    'window_size': 5,
    'target_name': TARGET_COL,
}

joblib.dump(artifact_m1, 'anomaly_detector.pkl')
print('✅ Đã xuất thành công file: anomaly_detector.pkl')